# Ostoskärrydatan tutkimus (03) - Laajennettu analyysi

## Tietovaraston lataus

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from datetime import datetime, timedelta
import warnings

warnings.filterwarnings('ignore')

# Asetetaan visuaalinen tyyli
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (14, 8)
plt.rcParams['font.size'] = 12

# Tarkistetaan datakansiot
raw_data_path = Path('data/raw')
processed_data_path = Path('data/processed')

print(f"Raw data path: {raw_data_path}")
print(f"Processed data path: {processed_data_path}")
print(f"\nRaw files: {list(raw_data_path.glob('*')) if raw_data_path.exists() else 'Path does not exist'}")

## Datan lataus

In [ ]:
# Ladataan raakadata CSV-tiedostoista
raw_file = 'data/raw/node_3224.csv'  # Voit muuttaa tätä

try:
    df = pd.read_csv(raw_file)
    print(f"✓ Data ladattu onnistuneesti: {raw_file}")
    print(f"\n📊 Datan koko: {df.shape[0]:,} riviä, {df.shape[1]} saraketta")
    print(f"\n📋 Sarakkeet: {list(df.columns)}")
except FileNotFoundError:
    print(f"✗ Virhe: Tiedostoa {raw_file} ei löydy.")
    print("Varmista, että data on ladattu data/raw/ -kansioon.")

## Datan perustietojen tarkastelu

In [ ]:
# Ensimäinen tarkastelu
print("="*60)
print(" DATAN PERUSTIEDOT ")
print("="*60)
print(f"\n📊 Datan koko: {df.shape[0]:,} riviä, {df.shape[1]} saraketta")

print("\n📋 Sarakkeet ja tyypit:")
print(df.dtypes)

print("\n❓ PUUTTUVAT ARVOT:")
null_counts = df.isnull().sum()
null_percentages = (null_counts / len(df)) * 100
null_df = pd.DataFrame({'Missing': null_counts, 'Percentage': null_percentages})
print(null_df[null_df['Missing'] > 0].sort_values('Missing', ascending=False))

print("\n📈 PERUSTILASTOT:")
print(df.describe())

## Datan siivous

In [ ]:
# Poistetaan rivit, joissa on puuttuvia arvoja kriittisissä sarakkeissa
critical_columns = ['node_id', 'timestamp', 'x', 'y', 'z']
available_columns = [col for col in critical_columns if col in df.columns]

df_initial = df.copy()
df_cleaned = df.copy()
rows_dropped = 0

if available_columns:
    rows_dropped = len(df) - len(df.dropna(subset=available_columns))
    df_cleaned = df.dropna(subset=available_columns)
    print(f"🗑️  Rivit poistettu puuttuvista arvoista: {rows_dropped:,} ({rows_dropped/len(df)*100:.2f}%)")
else:
    print("⚠️  Huom: Kriittisiä sarakkeita ei löydy, ei suoritettu puuttuvien arvojen poistoa.")

# Valinnainen: Poistetaan sarakkeet, joissa on enemmän kuin 50% puuttuvia arvoja
threshold = 0.5 * len(df_cleaned)
cols_to_drop = df_cleaned.columns[df_cleaned.isnull().sum() > threshold].tolist()
if cols_to_drop:
    df_cleaned = df_cleaned.drop(columns=cols_to_drop)
    print(f"🗑️  Sarakkeet poistettu: {cols_to_drop}")

print(f"\n✅ Siivottu data: {df_cleaned.shape[0]:,} riviä, {df_cleaned.shape[1]} saraketta")

## 📊 Edistynyt tilastot ja analyysi

In [ ]:
# Asetetaan koordinaatit datetime-objekteiksi
df_cleaned['timestamp'] = pd.to_datetime(df_cleaned['timestamp'])
df_cleaned = df_cleaned.sort_values('timestamp').reset_index(drop=True)

print("="*60)
print(" 🌍 SPATIAALISET TILASTOT ")
print("="*60)

# X-akselin tilastot
print(f"\n📈 X-akseli (Pohjoinen-Etelä):")
print(f"  - Min: {df_cleaned['x'].min():.2f}")
print(f"  - Max: {df_cleaned['x'].max():.2f}")
print(f"  - Keskiarvo: {df_cleaned['x'].mean():.2f}")
print(f"  - Kuvaaja: {df_cleaned['x'].median():.2f}")
print(f"  - Keskihajonta: {df_cleaned['x'].std():.2f}")

# Y-akselin tilastot
print(f"\n📈 Y-akseli (Itä-Länsi):")
print(f"  - Min: {df_cleaned['y'].min():.2f}")
print(f"  - Max: {df_cleaned['y'].max():.2f}")
print(f"  - Keskiarvo: {df_cleaned['y'].mean():.2f}")
print(f"  - Kuvaaja: {df_cleaned['y'].median():.2f}")
print(f"  - Keskihajonta: {df_cleaned['y'].std():.2f}")

# Z-akselin tilastot
print(f"\n📈 Z-akseli (Ylähaut):")
print(f"  - Min: {df_cleaned['z'].min():.2f}")
print(f"  - Max: {df_cleaned['z'].max():.2f}")
print(f"  - Keskiarvo: {df_cleaned['z'].mean():.2f}")
print(f"  - Kuvaaja: {df_cleaned['z'].median():.2f}")
print(f"  - Keskihajonta: {df_cleaned['z'].std():.2f}")

# Q-arvojen tilastot
if 'q' in df_cleaned.columns:
    print(f"\n📈 Q-arvot (Lisäarvo):")
    print(f"  - Min: {df_cleaned['q'].min():.2f}")
    print(f"  - Max: {df_cleaned['q'].max():.2f}")
    print(f"  - Keskiarvo: {df_cleaned['q'].mean():.2f}")
    print(f"  - Kuvaaja: {df_cleaned['q'].median():.2f}")
    print(f"  - Keskihajonta: {df_cleaned['q'].std():.2f}")

# Aikaväli
time_range = df_cleaned['timestamp'].max() - df_cleaned['timestamp'].min()
days = time_range.days
hours = (time_range.seconds / 3600)
print(f"\n⏰ Kattava aikaväli:")
print(f"  - Aloitus: {df_cleaned['timestamp'].min()}")
print(f"  - Lopetus: {df_cleaned['timestamp'].max()}")
print(f"  - Koko: {days} päivää {hours:.1f} tuntia")

## ⏰ Aikaan perustuva analyysi

In [ ]:
# Aikapohjaiset tilastot
df_cleaned['hour'] = df_cleaned['timestamp'].dt.hour
df_cleaned['day'] = df_cleaned['timestamp'].dt.day
df_cleaned['weekday'] = df_cleaned['timestamp'].dt.day_name()
df_cleaned['date'] = df_cleaned['timestamp'].dt.date

print("="*60)
print(" ⏰ Aikapohjaiset Tilastot ")
print("="*60)

print(f"\n📅 Päivät:")
print(f"  - Koko ajanjakso: {len(df_cleaned['date'].unique())} päivää")
print(f"  - Huipun päivä: {df_cleaned.groupby('date').size().idxmax()} ({df_cleaned.groupby('date').size().max():,} kohtaa)")
print(f"  - Eniten kohtia päivässä: {df_cleaned.groupby('date').size().max():,} kohtaa")
print(f"  - Vähiten kohtia päivässä: {df_cleaned.groupby('date').size().min():,} kohtaa")

print(f"\n🕒 Kuukauden jakautuminen:")
df_cleaned['month'] = df_cleaned['timestamp'].dt.month
monthly_counts = df_cleaned.groupby('month').size()
for month, count in monthly_counts.items():
    print(f"  - Kuukausi {month}: {count:,} kohtaa ({count/len(df_cleaned)*100:.2f}%)")

print(f"\n🕐 Tuntikohtainen jakautuminen:")
hourly_counts = df_cleaned.groupby('hour').size()
print(f"  - Huipun tunti: {hourly_counts.idxmax()}:00 ({hourly_counts.max():,} kohtaa)")
print(f"  - Eniten kohtia tuntia: {hourly_counts.max():,} kohtaa")
print(f"  - Vähiten kohtia tuntia: {hourly_counts.min():,} kohtaa")
print(f"  - Kuukauden keskimääräisiä kohtia tunnissa: {hourly_counts.mean():.1f}")

## 📏 Etäisyys-analyysi

In [ ]:
# Lasketaan koordinaatteihin perustuvat etäisyydet
print("="*60)
print(" 📏 ETÄISYYS-ANALYYSI ")
print("="*60)

# Lasketaan per rivi etäisyys aiempaan koordinaattiin (simulaatio)
df_cleaned['dist_x'] = df_cleaned['x'].diff().abs()
df_cleaned['dist_y'] = df_cleaned['y'].diff().abs()
df_cleaned['dist_z'] = df_cleaned['z'].diff().abs()

# Hypoteettinen fyysinen etäisyys (0.1m = 1 yksikkö koordinaatissa)
df_cleaned['total_distance'] = (df_cleaned['dist_x'] + df_cleaned['dist_y'] + df_cleaned['dist_z']) * 0.1

print(f"\n📏 Liikkeen pituudet:")
print(f"  - X-akseli: keskimäärin {df_cleaned['dist_x'].mean():.2f} yksikköä ({df_cleaned['dist_x'].mean()*0.1:.2f} m)")
print(f"  - Y-akseli: keskimäärin {df_cleaned['dist_y'].mean():.2f} yksikköä ({df_cleaned['dist_y'].mean()*0.1:.2f} m)")
print(f"  - Z-akseli: keskimäärin {df_cleaned['dist_z'].mean():.2f} yksikköä ({df_cleaned['dist_z'].mean()*0.1:.2f} m)")
print(f"  - Yhteensä: keskimäärin {df_cleaned['total_distance'].mean():.2f} m / liike")
print(f"  - Suurin liike: {df_cleaned['total_distance'].max():.2f} m")
print(f"  - Kuvaaja liike: {df_cleaned['total_distance'].median():.2f} m")

print(f"\n🚶 Liikesekvenssit:")
df_cleaned['moves_per_session'] = (df_cleaned['total_distance'] > 0.1).astype(int)
total_moves = df_cleaned['moves_per_session'].sum()
avg_moves = df_cleaned.groupby('node_id')['moves_per_session'].sum().mean()
print(f"  - Yhteensä liikkeitä: {total_moves:,}")
print(f"  - Keskimääräinen liikkeitä sekvenssissä: {avg_moves:.1f}")

## 🏪 Sekvenssi-analyysi (Shopping Session)

In [ ]:
# Sekvensseihin liittyvät tilastot
print("="*60)
print(" 🏪 SEKVENSSI-ANALYYSI ")
print("="*60)

print(f"\n👥 Kärryt:")
unique_users = df_cleaned['node_id'].nunique()
print(f"  - Käyttäjien määrä: {unique_users}")
print(f"  - Kaikkia kohtia Kärryä kohden: {len(df_cleaned) / unique_users:,.1f}")

# Sekvensseihin liittyvät aikavälit
session_durations = df_cleaned.groupby('node_id').apply(
    lambda x: (x['timestamp'].max() - x['timestamp'].min()).total_seconds()
).dropna()

print(f"\n⏱️ Sekvenssien kestot:")
print(f"  - Keskimääräinen kesto: {session_durations.mean():.1f} sekuntia ({session_durations.mean()/60:.1f} minuuttia)")
print(f"  - Pisin sekvenssi: {session_durations.max():.1f} sekuntia ({session_durations.max()/60:.1f} minuuttia)")
print(f"  - Lyhin sekvenssi: {session_durations.min():.1f} sekuntia ({session_durations.min()/60:.1f} minuuttia)")
print(f"  - Kuvaaja: {session_durations.median():.1f} sekuntia")

# Kohtausten tiheys
avg_interval = df_cleaned.groupby('node_id')['timestamp'].diff().mean()
min_interval = df_cleaned.groupby('node_id')['timestamp'].diff().abs().min()

print(f"\n⏰ Kohtausten tiheys:")
print(f"  - Keskimääräinen väli: {avg_interval.total_seconds():.1f} sekuntia")
print(f"  - Tihein kohtaustahti: {min_interval.total_seconds():.1f} sekuntia")

# Paikkajakauma
print(f"\n📍 Kohtausten paikkajakauma:")
location_stats = df_cleaned.groupby(['x', 'y', 'z']).size().sort_values(ascending=False)
top_locations = location_stats.head(10)
for (x, y, z), count in top_locations.items():
    print(f"  - Pos {x:.1f}, {y:.1f}, {z:.1f}: {count:,} kohtaa")

## 📊 Q-arvojen analyysi

In [ ]:
if 'q' in df_cleaned.columns:
    print("="*60)
    print(" 📊 Q-ARVOJEN ANALYYSI ")
    print("="*60)

    # Q-arvon perustilastot
    q_stats = df_cleaned['q'].describe()
    print(f"\nQ-arvon perustilastot:")
    print(f"  - Min: {q_stats['min']}")
    print(f"  - Max: {q_stats['max']}")
    print(f"  - Keskiarvo: {q_stats['mean']:.2f}")
    print(f"  - Kuvaaja: {q_stats['50%']:.2f}")
    print(f"  - Kuviosta: {q_stats['75%']:.2f}")
    print(f"  - Keskihajonta: {q_stats['std']:.2f}")

    # Q-arvon jakauma aikapohjaisesti
    print(f"\n⏰ Q-arvon kehitys päivittäin:")
    daily_q = df_cleaned.groupby('date')['q'].agg(['mean', 'std', 'min', 'max'])
    print(daily_q.head(10))

    # Q-arvon korrelaatio ajan kanssa
    q_time_corr = df_cleaned[['q', 'hour']].corr().iloc[0, 1]
    print(f"\n🔗 Q-arvon ja ajan korrelaatio: {q_time_corr:.3f}")

## 🗺️ X-Y Heatmap visualisointi

In [ ]:
# X-Y Heatmap
print("\n" + "="*60)
print(" 🗺️ X-Y HEATMAP ")
print("="*60)

# Lasketaan kohtausten tiheys
df_xy = df_cleaned[['x', 'y']].copy()

fig, axes = plt.subplots(2, 2, figsize=(16, 12))
fig.suptitle('X-Y Koordinaattien Jakautuminen', fontsize=16, fontweight='bold')

# 1. Perus scatter plot
axes[0, 0].scatter(df_xy['x'], df_xy['y'], alpha=0.3, s=1, c='blue')
axes[0, 0].set_xlabel('X-akseli (Pohjoinen-Etelä)')
axes[0, 0].set_ylabel('Y-akseli (Itä-Länsi)')
axes[0, 0].set_title('Kaikki kohtaukset')
axes[0, 0].grid(True, alpha=0.3)

# 2. Heatmap (binning)
x_bins = np.linspace(df_xy['x'].min(), df_xy['x'].max(), 50)
y_bins = np.linspace(df_xy['y'].min(), df_xy['y'].max(), 50)
heatmap_data, xedges, yedges = np.histogram2d(df_xy['x'], df_xy['y'], bins=[x_bins, y_bins])

im = axes[0, 1].imshow(heatmap_data.T, origin='lower', cmap='YlOrRd', aspect='auto', extent=[x_bins[0], x_bins[-1], y_bins[0], y_bins[-1]])
axes[0, 1].set_xlabel('X-akseli')
axes[0, 1].set_ylabel('Y-akseli')
axes[0, 1].set_title('Kohtausten tiheys (Heatmap)')
plt.colorbar(im, ax=axes[0, 1], label='Kohtausten määrä')

# 3. Top 10 sijainnit
location_counts = df_xy.groupby(['x', 'y']).size().sort_values(ascending=False).head(10)
axes[1, 0].barh(range(len(location_counts)), location_counts.values, color='steelblue')
axes[1, 0].set_yticks(range(len(location_counts)))
axes[1, 0].set_yticklabels([f"({x:.1f}, {y:.1f})" for (x, y) in location_counts.index])
axes[1, 0].set_xlabel('Kohtausten määrä')
axes[1, 0].set_title('Top 10 Sijainti')
axes[1, 0].invert_yaxis()

# 4. X- ja Y-akselien jakauma
axes[1, 1].hist(df_xy['x'], bins=30, alpha=0.5, label='X', color='blue')
axes[1, 1].hist(df_xy['y'], bins=30, alpha=0.5, label='Y', color='orange')
axes[1, 1].set_xlabel('Arvo')
axes[1, 1].set_ylabel('Frekvenssi')
axes[1, 1].set_title('X- ja Y-akselien jakaumat')
axes[1, 1].legend()
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("✅ X-Y heatmap luotu onnistuneesti!")

## 🌐 3D X-Y-Z Koordinaattien Scatter Plot

In [ ]:
# 3D Scatter plot
print("\n" + "="*60)
print(" 🌐 3D KOORDINAATTIEN VISUALISATION ")
print("="*60)

from mpl_toolkits.mplot3d import Axes3D

fig = plt.figure(figsize=(14, 10))
ax = fig.add_subplot(111, projection='3d')

# Otsikko ja akselien nimet
ax.set_title('3D Koordinaattien Jakauma', fontsize=16, fontweight='bold')
ax.set_xlabel('X-akseli (Pohjoinen-Etelä)')
ax.set_ylabel('Y-akseli (Itä-Länsi)')
ax.set_zlabel('Z-akseli (Ylähaut)')

# Rajat
ax.set_xlim(df_cleaned['x'].min(), df_cleaned['x'].max())
ax.set_ylim(df_cleaned['y'].min(), df_cleaned['y'].max())
ax.set_zlim(df_cleaned['z'].min(), df_cleaned['z'].max())

# Scatter plot
# Näytetään vain jokin osa datasta suuruisen datan vuoksi
sample_size = min(10000, len(df_cleaned))
df_sample = df_cleaned.sample(sample_size, random_state=42)

q_data = df_sample['q'] if 'q' in df_sample.columns else 'blue'

scatter = ax.scatter(
    df_sample['x'], 
    df_sample['y'], 
    df_sample['z'], 
    c=q_data, 
    cmap='viridis' if 'q' in df_sample.columns else None, 
    alpha=0.6, 
    s=2,
    edgecolors='none'
)

if 'q' in df_sample.columns:
    cbar = fig.colorbar(scatter, ax=ax, pad=0.1)
    cbar.set_label('Q-arvo', rotation=270, labelpad=20)

plt.tight_layout()
plt.show()

print(f"✅ 3D scatter plot luotu! Näytetään {sample_size:,} / {len(df_cleaned):,} kohdetta.")

## 📈 Aikasarja-analyysi

In [ ]:
# Aikasarja-analyysi
print("\n" + "="*60)
print(" 📈 Aikasarja-analyysi ")
print("="*60)

fig, axes = plt.subplots(3, 2, figsize=(16, 15))
fig.suptitle('Aikasarja-analyysi', fontsize=16, fontweight='bold')

# 1. Kohtauksien määrä päivittäin
daily_counts = df_cleaned.groupby('date').size()
axes[0, 0].plot(daily_counts.index, daily_counts.values, linewidth=2, color='blue')
axes[0, 0].set_xlabel('Päivämäärä')
axes[0, 0].set_ylabel('Kohtauksien määrä')
axes[0, 0].set_title('Kohtauksien määrä päivittäin')
axes[0, 0].grid(True, alpha=0.3)
axes[0, 0].tick_params(axis='x', rotation=45)

# 2. Kohtauksien määrä tunnissa
hourly_counts = df_cleaned.groupby('hour').size()
axes[0, 1].bar(hourly_counts.index, hourly_counts.values, color='coral', alpha=0.7)
axes[0, 1].set_xlabel('Tunti (0-23)')
axes[0, 1].set_ylabel('Kohtauksien määrä')
axes[0, 1].set_title('Kohtauksien määrä tunnissa')
axes[0, 1].grid(True, alpha=0.3, axis='y')

# 3. Kohtauksien määrä viikonpäivittäin
weekday_counts = df_cleaned.groupby('weekday').size()
weekday_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
weekday_counts = weekday_counts.reindex(weekday_order)
axes[1, 0].plot(weekday_counts.index, weekday_counts.values, marker='o', linewidth=2, color='green')
axes[1, 0].set_xlabel('Viikonpäivä')
axes[1, 0].set_ylabel('Kohtauksien määrä')
axes[1, 0].set_title('Kohtauksien määrä viikonpäivittäin')
axes[1, 0].grid(True, alpha=0.3)

# 4. Q-arvon kehitys aikaa seuraten
if 'q' in df_cleaned.columns:
    q_over_time = df_cleaned.groupby('date')['q'].mean()
    axes[1, 1].plot(q_over_time.index, q_over_time.values, linewidth=2, color='purple')
    axes[1, 1].set_xlabel('Päivämäärä')
    axes[1, 1].set_ylabel('Q-arvon keskiarvo')
    axes[1, 1].set_title('Q-arvon kehitys aikaa seuraten')
    axes[1, 1].grid(True, alpha=0.3)
    axes[1, 1].tick_params(axis='x', rotation=45)
else:
    axes[1, 1].set_title('Ei Q-arvoa saatavilla')

# 5. Liikekertojen määrä tunnissa
moves_per_hour = df_cleaned.groupby(['hour', 'date']).size().groupby('hour').mean()
axes[2, 0].plot(moves_per_hour.index, moves_per_hour.values, linewidth=2, color='orange')
axes[2, 0].set_xlabel('Tunti')
axes[2, 0].set_ylabel('Keskimääräinen liikekertoja')
axes[2, 0].set_title('Keskimääräiset liikekerrat tunnissa')
axes[2, 0].grid(True, alpha=0.3)

# 6. Etäisyyden kehitys aikaa seuraten (viikkoittain)
weekly_dist = df_cleaned.groupby('date')['total_distance'].sum()
axes[2, 1].plot(weekly_dist.index, weekly_dist.values, linewidth=2, color='teal')
axes[2, 1].set_xlabel('Päivämäärä')
axes[2, 1].set_ylabel('Yhteinen etäisyys (m)')
axes[2, 1].set_title('Yhteinen etäisyys viikkoittain')
axes[2, 1].grid(True, alpha=0.3)
axes[2, 1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

print("✅ Aikasarja-analyysi valmis!")

## 📊 Q-arvojen jakautuminen

In [ ]:
if 'q' in df_cleaned.columns:
    print("\n" + "="*60)
    print(" 📊 Q-ARVOJEN JAKAUTUMINEN ")
    print("="*60)

    fig, axes = plt.subplots(2, 2, figsize=(16, 12))
    fig.suptitle('Q-arvojen jakaumat', fontsize=16, fontweight='bold')

    # 1. Histogrammi
    axes[0, 0].hist(df_cleaned['q'], bins=50, color='skyblue', edgecolor='black', alpha=0.7)
    axes[0, 0].set_xlabel('Q-arvo')
    axes[0, 0].set_ylabel('Frekvenssi')
    axes[0, 0].set_title('Q-arvon histogrammi')
    axes[0, 0].grid(True, alpha=0.3, axis='y')

    # 2. KDE plot
    sns.kdeplot(data=df_cleaned['q'], ax=axes[0, 1], fill=True, alpha=0.6, color='orange')
    axes[0, 1].set_xlabel('Q-arvo')
    axes[0, 1].set_ylabel('Tiheys')
    axes[0, 1].set_title('Q-arvon KDE plot')
    axes[0, 1].grid(True, alpha=0.3)

    # 3. Päivittäinen keskiarvo
    daily_q_mean = df_cleaned.groupby('date')['q'].mean()
    axes[1, 0].plot(daily_q_mean.index, daily_q_mean.values, linewidth=2, color='green')
    axes[1, 0].axhline(y=daily_q_mean.mean(), color='red', linestyle='--', label=f'Keskiarvo: {daily_q_mean.mean():.2f}')
    axes[1, 0].set_xlabel('Päivämäärä')
    axes[1, 0].set_ylabel('Q-arvon keskiarvo')
    axes[1, 0].set_title('Q-arvon päivittäinen keskiarvo')
    axes[1, 0].legend()
    axes[1, 0].grid(True, alpha=0.3)
    axes[1, 0].tick_params(axis='x', rotation=45)

    # 4. Q-arvon tuntikohtainen jakauma
    sns.boxplot(x='hour', y='q', data=df_cleaned, ax=axes[1, 1], palette='pastel')
    axes[1, 1].set_xlabel('Tunti')
    axes[1, 1].set_ylabel('Q-arvo')
    axes[1, 1].set_title('Q-arvon jakauma tuntikohtaisesti')
    axes[1, 1].grid(True, alpha=0.3, axis='y')

    plt.tight_layout()
    plt.show()

    print("✅ Q-arvojen jakaumat valmis!")
else:
    print("⚠️ Saraketta 'q' ei löydy, ohitetaan Q-arvojen jakautumisen piirtäminen.")

## 🛤️ Trajektorioiden visualisointi

In [ ]:
# Trajektorioiden visualisointi
print("\n" + "="*60)
print(" 🛤️ TRAJECTORY VISUALISATION ")
print("="*60)

# Valitaan satunnainen kärry näyttämiseen
random_user = df_cleaned['node_id'].sample(1).values[0]
user_data = df_cleaned[df_cleaned['node_id'] == random_user].sort_values('timestamp')

print(f"\n👤 Näytetään kärryn {random_user} liikkeitä")
print(f"   - Sekvenssi: {len(user_data)} kohtaa")
print(f"   - Aikaväli: {(user_data['timestamp'].max() - user_data['timestamp'].min()).total_seconds():.1f} sekuntia")
print(f"   - Yhteinen matka: {user_data['total_distance'].sum():.2f} m")

fig, axes = plt.subplots(2, 2, figsize=(16, 12))
fig.suptitle(f'Trajectory analyysi - Kärry {random_user}', fontsize=16, fontweight='bold')

# 1. Trajektorioiden 2D plot
axes[0, 0].plot(user_data['x'], user_data['y'], linewidth=2, color='blue', marker='o', markersize=3, alpha=0.6)
axes[0, 0].set_xlabel('X-akseli')
axes[0, 0].set_ylabel('Y-akseli')
axes[0, 0].set_title('Trajektorioiden 2D -kulma')
axes[0, 0].grid(True, alpha=0.3)

# 2. Trajektorioiden 3D plot
axes[0, 1].remove() # Poistetaan tieltä 2D akseli, jotta 3D mahtuu
ax_3d = fig.add_subplot(2, 2, 2, projection='3d')
ax_3d.plot(user_data['x'], user_data['y'], user_data['z'], linewidth=2, color='green', marker='o', markersize=3, alpha=0.6)
ax_3d.set_xlabel('X')
ax_3d.set_ylabel('Y')
ax_3d.set_zlabel('Z')
ax_3d.set_title('Trajektorioiden 3D -kulma')

# 3. Ajan ja etäisyyden yhteys
axes[1, 0].plot(user_data['timestamp'], user_data['total_distance'].cumsum(), linewidth=2, color='orange')
axes[1, 0].set_xlabel('Aika')
axes[1, 0].set_ylabel('Kumulatiivinen etäisyys (m)')
axes[1, 0].set_title('Etäisyyden kehitys ajan funktiona')
axes[1, 0].grid(True, alpha=0.3)
axes[1, 0].tick_params(axis='x', rotation=45)

# 4. Liikekertojen tiheys ajan mukaan
axes[1, 1].plot(user_data['timestamp'], range(len(user_data)), linewidth=2, color='purple')
axes[1, 1].set_xlabel('Aika')
axes[1, 1].set_ylabel('Liikekertojen järjestysnumero')
axes[1, 1].set_title('Liikekertojen aikajana')
axes[1, 1].grid(True, alpha=0.3)
axes[1, 1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

print("✅ Trajektorioiden visualisointi valmis!")

## 📝 Yhteenveto ja keskeiset havainnot

In [ ]:
# Keskeisten havaintojen tiivistelmä
print("\n" + "="*60)
print(" 📊 KESKEISET HAVAINNOT ")
print("="*60)

summary = f"""
DATAN YHTEENVETO:
- Total records: {len(df_cleaned):,}
- Unique users: {df_cleaned['node_id'].nunique()}
- Time range: {df_cleaned['timestamp'].min()} to {df_cleaned['timestamp'].max()}
- Duration: {df_cleaned['timestamp'].max() - df_cleaned['timestamp'].min()}
- Average per user: {len(df_cleaned) / df_cleaned['node_id'].nunique():,.1f}
- X-range: {df_cleaned['x'].min():.1f} to {df_cleaned['x'].max():.1f}
- Y-range: {df_cleaned['y'].min():.1f} to {df_cleaned['y'].max():.1f}
- Z-range: {df_cleaned['z'].min():.1f} to {df_cleaned['z'].max():.1f}
- Total distance moved: {df_cleaned['total_distance'].sum():,.2f} m
- Average distance per move: {df_cleaned['total_distance'].mean():.2f} m
- Q-value average: {df_cleaned['q'].mean() if 'q' in df_cleaned.columns else 'N/A'}
"""

print(summary)

print("\n✅ Laajennettu analyysi valmis!")
print("\nSeuraavat vaiheet:")
print("1. Tallenna df_cleaned muuttuja jatkokäyttöön")
print("2. Tee lisäanalyyseja tarvittaessa")
print("3. Tallenna tulokset data/processed -hakemistoon")